In [3]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [16]:
import sys
import os

# Go up one level to the project root so Python can find 'moogp' and 'ogp'
sys.path.append(os.path.abspath('..'))

from moogp.model import init_phi
from moogp.datasets import generate_forrester_data

import numpy as np

In [17]:
data = generate_forrester_data(n=100, seed=67,error_per_output=[1, 1, 1])

In [18]:
X = data['X']
X_scaled = data['X_scaled']
Y = data['y']

In [62]:
Phi, d = init_phi(Y,3,100)

In [31]:
import tensorflow as tf

In [71]:
def tf_init_phi(y,q,n,p):
    """
    Initialization of orthogonal basis, computed with singular value decomposition.
    """
    
    singvals, left_u, _ = tf.linalg.svd(y, full_matrices=False)

    if (q is None) and (var_threshold is None):
        q = p
    elif (q is None) and (var_threshold is not None):
        cumvar = tf.cumsum(singvals ** 2) / tf.reduce_sum(singvals ** 2)
        q = int(tf.argmax(cumvar > var_threshold) + 1)

    assert left_u.shape[1] == min(n, p)
    singvals = singvals[:q]

    # Compute phi and diag_D
    phi = left_u[:, :q] * tf.sqrt(tf.cast(n, tf.float64)) / singvals
    diag_D = tf.reduce_sum(phi ** 2, axis=0)

    g = tf.matmul(phi, y, transpose_a=True)
    return g, phi, diag_D, q

In [72]:
g, phi, diag_D, q = tf_init_phi(Y,3,100,3)

In [69]:
np.allclose(abs(Phi), abs(np.asarray(phi)))

True

In [70]:
np.allclose(d, diag_D)

True

In [73]:
phi

<tf.Tensor: shape=(100, 3), dtype=float64, numpy=
array([[-3.36468297e-03, -1.84254785e-02,  6.66888379e-01],
       [-6.77221897e-03, -1.80487735e-02,  2.93008454e-01],
       [-7.46438763e-03,  4.17424328e-03,  1.06865150e+00],
       [-1.27431555e-03, -5.56691027e-02, -5.26978894e-01],
       [-4.18703116e-03, -1.74491945e-02,  6.10703941e-01],
       [-5.85674428e-03, -1.71447396e-02,  4.32109724e-01],
       [-5.44518377e-03, -4.39617237e-03,  9.69230935e-01],
       [-7.06803846e-04, -5.46619375e-02, -4.23570154e-01],
       [-8.50012483e-04, -5.67509714e-02, -5.20220630e-01],
       [-5.98030523e-03, -1.74622250e-02,  4.05819066e-01],
       [-3.95876330e-04, -5.62949558e-02, -4.50926668e-01],
       [-7.81229162e-03, -1.69554907e-02,  2.16508324e-01],
       [-7.77854912e-03, -2.40401110e-02, -5.20675314e-02],
       [-1.94496207e-02,  1.69381517e-03, -3.92713471e-01],
       [-8.46364609e-03, -1.61924248e-02,  1.71613681e-01],
       [-6.27721852e-03, -1.80009466e-02,  3.51263